# Clase en vivo 3 · Deep Learning y modelos avanzados

**La pregunta de la clase:** ¿cuándo una red neuronal mejora una solución y cuándo solo agrega costo y opacidad?

Vamos a **medirlo** con NovaRetail (datos sintéticos generados por código; ningún resultado representa una situación real de ARCA). Antes de cada resultado importante hay una **predicción**: anotala o decila en voz alta antes de correr la celda.

| Parte | Secciones |
| --- | --- |
| Fundamentos con código | 1 a 7: datos, anatomía de una red, entrenamiento, la comparación en dos casos y sobreajuste |
| Arquitecturas | 8 a 12: convolución, secuencias, atención, modelos generativos y cómo elegir |
| Errores y cierre | 13 y 14: el laboratorio de errores y la respuesta a la pregunta |

Corré las celdas **en orden**. Cada sección tarda segundos, salvo la preparación.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = True          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## 0 · Preparación

Esta celda carga las librerías, genera NovaRetail (caso 1) y define las funciones. **Además lanza en segundo plano dos entrenamientos pesados** que vamos a necesitar en la sección 7, para que estén listos cuando lleguemos. No hace falta leerla.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

VIOLETA, VERDE, NARANJA, GRIS, ROJO = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8", "#E5484D"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.3f}".format

COLUMNAS = ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
            "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
            "satisfaccion", "motivo_baja", "abandono_90d"]


def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


import re, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
try:
    from xgboost import XGBClassifier
    def modelo_arboles(semilla): return XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=semilla, n_jobs=1, eval_metric="logloss")
    NOMBRE_ARBOLES = "XGBoost (100 árboles, prof. 3)"
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    def modelo_arboles(semilla): return HistGradientBoostingClassifier(max_depth=3, random_state=semilla)
    NOMBRE_ARBOLES = "Boosting de scikit-learn (XGBoost no está instalado)"

NUM = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion", "satisfaccion_faltante"]
NUEVAS = ["ciclos_sin_comprar", "gasto_anual", "contactos_por_compra"]
CAT = ["canal", "segmento", "region"]

def preparar(caso=1):
    """Caso 1: NovaRetail tal cual. Caso 2: los mismos clientes, tres variables nuevas y otra regla de abandono (umbral, interacción y efectos que no son una línea)."""
    df = generar_novaretail().drop_duplicates("id_cliente").reset_index(drop=True)
    df.loc[(df.edad < 18) | (df.edad > 100), "edad"] = np.nan
    df["satisfaccion_faltante"] = df.satisfaccion.isna().astype(int)
    c = df.compras_12m
    df["ciclos_sin_comprar"] = df.dias_desde_ultima_compra * np.maximum(c, 1) / 365
    df["gasto_anual"] = np.where(c == 0, 0.0, df.ticket_promedio * c)
    df["contactos_por_compra"] = df.contactos_soporte_90d / (c + 1)
    if caso == 2:
        azar = np.random.default_rng(2026)                       # flujo aparte: el caso 1 no cambia
        gasto = df.gasto_anual.fillna(df.gasto_anual.median())
        z = (-2.8 + 0.55 * (df.segmento == "Esencial") - 0.35 * (df.segmento == "Premium") + 0.35 * (df.dias_desde_ultima_compra - 60) / 40
             + 2.6 * (df.ciclos_sin_comprar > 2.0) + 2.4 * ((df.usa_app == 0) & (df.contactos_soporte_90d >= 3))
             + 1.6 * ((df.edad < 25) | (df.edad > 65)) - 1.2 * (gasto > 600))
        df["abandono_90d"] = (azar.random(len(df)) < 1 / (1 + np.exp(-z))).astype(int)
    return df

def comparar(caso=1, prevalencia=None, semillas=5, con_nuevas=None):
    """Compara regresión, red densa pequeña y árboles con la misma partición y la misma métrica.
    caso: 1 o 2 · prevalencia: por ejemplo 0.005 para dejar solo 0,5 % de casos positivos · con_nuevas: por defecto, False en el caso 1 y True en el caso 2."""
    df = preparar(caso)
    if con_nuevas is None: con_nuevas = (caso == 2)
    if prevalencia:
        neg, pos = df[df.abandono_90d == 0], df[df.abandono_90d == 1]
        n_pos = int(round(prevalencia * len(neg) / (1 - prevalencia))); df = pd.concat([neg, pos.sample(n_pos, random_state=0)]).reset_index(drop=True)
    num = NUM + (NUEVAS if con_nuevas else [])
    X, y = df[num + CAT], df["abandono_90d"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    def armar(m): return Pipeline([("prep", ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
                                                                ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])), ("modelo", m)])
    modelos = [("Regresión logística (L2)", lambda s: LogisticRegression(C=1.0, max_iter=3000), 1),
               ("Red densa (64, 32)", lambda s: MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, validation_fraction=0.15, n_iter_no_change=10, max_iter=500, random_state=s), semillas),
               (NOMBRE_ARBOLES, modelo_arboles, semillas)]
    filas = []
    for nombre, fab, reps in modelos:
        aucs, tops, tiempos = [], [], []
        for s in range(reps):
            t0 = time.time(); m = armar(fab(s)).fit(Xtr, ytr); tiempos.append(time.time() - t0)
            p = m.predict_proba(Xte)[:, 1]; aucs.append(roc_auc_score(yte, p))
            n_top = max(1, int(round(0.05 * len(yte)))); tops.append(np.asarray(yte)[np.argsort(-p)[:n_top]].mean())
        filas.append({"modelo": nombre, "AUC (media)": round(np.mean(aucs), 3), "AUC mín–máx": f"{min(aucs):.3f}–{max(aucs):.3f}", "aciertos en el 5 % de mayor riesgo": f"{100 * np.mean(tops):.0f} %", "entrena en (s)": round(np.mean(tiempos), 2)})
    tasa = f"{100 * y.mean():.1f}".replace(".", ","); n_clientes = f"{len(df):,}".replace(",", ".")
    print(f"Caso {caso} · {n_clientes} clientes · tasa de abandono {tasa} % · casos positivos en la prueba: {int(np.sum(yte))} · variables nuevas: {'sí' if con_nuevas else 'no'}")
    return pd.DataFrame(filas).set_index("modelo")



from sklearn.base import clone
from sklearn.metrics import log_loss
from concurrent.futures import ThreadPoolExecutor

def c3(x, d=3): return f"{x:.{d}f}".replace(".", ",")
def miles(n): return f"{n:,}".replace(",", ".")

def particion(caso=1, con_nuevas=None):
    df = preparar(caso)
    if con_nuevas is None: con_nuevas = (caso == 2)
    num = NUM + (NUEVAS if con_nuevas else [])
    X, y = df[num + CAT], df["abandono_90d"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    prep = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
                              ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
    return df, num, Xtr, Xte, ytr, yte, prep

def armar(modelo, prep): return Pipeline([("prep", clone(prep)), ("modelo", modelo)])

def matrices(caso=1, con_nuevas=None):
    df, num, Xtr, Xte, ytr, yte, prep = particion(caso, con_nuevas)
    A = prep.fit_transform(Xtr); B = prep.transform(Xte)
    A = A.toarray() if hasattr(A, "toarray") else A; B = B.toarray() if hasattr(B, "toarray") else B
    return A, B, np.asarray(ytr), np.asarray(yte), list(prep.get_feature_names_out())

df1, num1, Xtr1, Xte1, ytr1, yte1, prep1 = particion(1)
Xt1, Xe1, yt1, ye1, nombres1 = matrices(1)

def _red_grande(alpha):
    m = MLPClassifier(hidden_layer_sizes=(256, 256), alpha=alpha, max_iter=200, random_state=0).fit(Xt1, yt1)
    return roc_auc_score(yt1, m.predict_proba(Xt1)[:, 1]), roc_auc_score(ye1, m.predict_proba(Xe1)[:, 1])
_pool = ThreadPoolExecutor(max_workers=1)
GRANDES = {a: _pool.submit(_red_grande, a) for a in (0.0, 1.0, 3.0)}      # corren en segundo plano

print(f"Clientes: {miles(len(df1))}", f"· tasa de abandono: {c3(100 * df1.abandono_90d.mean(), 1)} %", f"· variables de entrada ya preparadas: {Xt1.shape[1]}")
print("Entrenamientos pesados lanzados en segundo plano: los usamos en la sección 7.")

## 1 · ¿Cuántos datos necesita cada modelo?

| | Machine Learning clásico | Deep Learning |
| --- | --- | --- |
| Variables | Se arman a mano | La red aprende sus propias representaciones |
| Datos | Funciona con pocos | Suele necesitar muchos más |
| Cómputo | Segundos en una computadora común | Puede pedir mucho más tiempo y hardware |
| Explicar | Coeficientes e importancias | Opaco |

La tabla dice «suele necesitar muchos más». Vamos a verlo: entrenamos con una parte cada vez mayor de los clientes y medimos el AUC en los mismos 2.000 de prueba.

> **Predicción:** con solo 250 clientes para entrenar, ¿qué modelo va a andar peor? ¿Y cuántos clientes hacen falta para que la red alcance a la regresión?

In [ ]:
tamaños = [250, 500, 1000, 2000, 4500]
azar = np.random.default_rng(0)
filas = []
for n in tamaños:
    idx = azar.choice(len(Xtr1), n, replace=False); a, b = Xtr1.iloc[idx], ytr1.iloc[idx]
    reg = roc_auc_score(yte1, armar(LogisticRegression(max_iter=3000), prep1).fit(a, b).predict_proba(Xte1)[:, 1])
    red = [roc_auc_score(yte1, armar(MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, validation_fraction=0.15, n_iter_no_change=10, max_iter=500, random_state=s), prep1).fit(a, b).predict_proba(Xte1)[:, 1]) for s in range(3)]
    arb = [roc_auc_score(yte1, armar(modelo_arboles(s), prep1).fit(a, b).predict_proba(Xte1)[:, 1]) for s in range(3)]
    filas.append({"clientes para entrenar": n, "Regresión": reg, "Red (media)": np.mean(red), "Red (mín)": min(red), "Red (máx)": max(red), "Árboles": np.mean(arb)})
curva = pd.DataFrame(filas).set_index("clientes para entrenar")
display(curva)

fig, ax = plt.subplots()
ax.plot(curva.index, curva["Regresión"], color=VIOLETA, lw=2.5, marker="o", label="Regresión logística")
ax.plot(curva.index, curva["Árboles"], color=VERDE, lw=2.5, marker="o", label="Árboles (XGBoost)")
ax.plot(curva.index, curva["Red (media)"], color=NARANJA, lw=2.5, marker="o", label="Red densa (media de 3 semillas)")
ax.fill_between(curva.index, curva["Red (mín)"], curva["Red (máx)"], color=NARANJA, alpha=0.2)
ax.axhline(0.5, color=GRIS, ls="--", lw=1); ax.text(260, 0.51, "azar", color=GRIS)
ax.set_xlabel("Clientes para entrenar"); ax.set_ylabel("AUC en la prueba"); ax.set_title("La red necesita más datos para empezar a aprender"); ax.legend(); plt.show()

**Qué mirar:** con 250 y 500 clientes la red está cerca del azar y la regresión ya funciona. Con 2.000 la red todavía es inestable: la banda naranja muestra lo que cambia según la semilla.

## 2 · Anatomía de una red: una neurona a mano

Una neurona **multiplica** cada entrada por un peso, **suma**, agrega un **sesgo** y aplica una **activación**. Con una sigmoide a la salida, una neurona sola es una regresión logística. Lo comprobamos con un cliente de la prueba.

> **Predicción:** si sumamos todos los términos de las 21 entradas y aplicamos la sigmoide, ¿qué número vamos a obtener? ¿Coincide con `predict_proba`?

In [ ]:
reg = LogisticRegression(C=1.0, max_iter=3000).fit(Xt1, yt1)
cliente = Xe1[0]; pesos = reg.coef_[0]; sesgo = reg.intercept_[0]
tres = ["num__dias_desde_ultima_compra", "num__compras_12m", "num__contactos_soporte_90d"]
print("Tres de las 21 entradas de este cliente:")
for nombre in tres:
    i = nombres1.index(nombre); print(f"  {nombre.split('__')[1]:<26} entrada {cliente[i]:+.2f} × peso {pesos[i]:+.2f} = {cliente[i] * pesos[i]:+.3f}")
z = float(cliente @ pesos + sesgo); activacion = 1 / (1 + np.exp(-z))
print(f"\nSuma de las 21 entradas + sesgo ({sesgo:+.2f}) = {z:+.3f}")
print(f"Sigmoide({z:+.3f}) = {activacion:.4f}   ← probabilidad de abandono")
print(f"predict_proba      = {reg.predict_proba(Xe1[:1])[0, 1]:.4f}")

In [ ]:
def parametros(entradas, capas):
    total, previo = 0, entradas
    for h in list(capas) + [1]: total += previo * h + h; previo = h
    return total

x = np.linspace(-4, 4, 200)
fig, ejes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (nombre, y) in zip(ejes, [("ReLU", np.maximum(0, x)), ("Sigmoide", 1 / (1 + np.exp(-x))), ("Tanh", np.tanh(x))]):
    ax.plot(x, y, color=VIOLETA, lw=3); ax.set_title(nombre); ax.axhline(0, color=GRIS, lw=0.6); ax.axvline(0, color=GRIS, lw=0.6)
plt.tight_layout(); plt.show()

entradas = Xt1.shape[1]
print(f"Parámetros con {entradas} entradas:")
for nombre, capas in [("Regresión logística", ()), ("Red (8)", (8,)), ("Red (16, 8)", (16, 8)), ("Red (64, 32)", (64, 32)), ("Red (128, 64, 32)", (128, 64, 32))]:
    print(f"  {nombre:<22} {miles(parametros(entradas, capas)):>7}")

**Qué mirar:** la neurona a mano coincide con `predict_proba`. Y cada capa que se agrega multiplica los parámetros: de 22 a más de 13.000.

## 3 · Entrenar una red: el ciclo y el learning rate

El entrenamiento repite cuatro pasos: **predecir**, **medir la pérdida**, **calcular el gradiente** y **ajustar los pesos**. El **learning rate** decide cuánto se mueve cada peso en cada paso. Lo hacemos a mano sobre una sola neurona, con tres tamaños de paso.

> **Predicción:** ¿qué pasa con la pérdida si el paso es muy grande? ¿Y si es muy chico?

In [ ]:
def entrenar(lr, epocas=120):
    w = np.zeros(Xt1.shape[1]); b = 0.0; historia = []
    for _ in range(epocas):
        p = np.clip(1 / (1 + np.exp(-(Xt1 @ w + b))), 1e-9, 1 - 1e-9)
        historia.append(-np.mean(yt1 * np.log(p) + (1 - yt1) * np.log(1 - p)))
        g = p - yt1; w -= lr * (Xt1.T @ g) / len(yt1); b -= lr * g.mean()
    return historia

pasos = [(0.01, "muy chico: 0,01", NARANJA), (0.5, "razonable: 0,5", VERDE), (12.0, "muy grande: 12", ROJO)]
fig, ax = plt.subplots()
for lr, etiqueta, color in pasos:
    h = entrenar(lr); ax.plot(range(1, 121), h, color=color, lw=2.4, label=f"paso {etiqueta}")
    print(f"paso {etiqueta:<18} pérdida final {c3(h[-1])} · pérdida máxima {c3(max(h))}")
ax.set_xlabel("Épocas"); ax.set_ylabel("Pérdida de entrenamiento"); ax.legend(); plt.show()

**Qué mirar:** el paso chico baja, pero a las 120 épocas todavía está en 0,49; el razonable se asienta en 0,31; el muy grande rebota y la pérdida llega a subir hasta 1,10.

## 4 · La comparación central · Caso 1: NovaRetail

Tres modelos, **misma partición y misma métrica** (AUC en 2.000 clientes de prueba): la regresión logística regularizada, una red densa pequeña y XGBoost. Las redes y XGBoost se entrenan con 5 semillas.

> **Predicción:** ¿gana la red? ¿Por cuánto?

In [ ]:
r1 = comparar(1)
display(r1)

modelos = list(r1.index); medias = r1["AUC (media)"].values
minimos = [float(t.split("–")[0]) for t in r1["AUC mín–máx"]]; maximos = [float(t.split("–")[1]) for t in r1["AUC mín–máx"]]
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, m in enumerate(modelos):
    ax.plot([minimos[i], maximos[i]], [i, i], color=GRIS, lw=2); ax.scatter([medias[i]], [i], s=110, color=VIOLETA if i == 0 else GRIS, zorder=3); ax.text(maximos[i] + 0.002, i, c3(medias[i]), va="center")
ax.axvline(medias[0], color=VIOLETA, ls="--", lw=1.2); ax.set_yticks(range(len(modelos))); ax.set_yticklabels(modelos); ax.invert_yaxis()
ax.set_xlim(0.815, 0.845); ax.set_xlabel("AUC en la prueba (media y rango entre semillas)"); ax.set_title("Caso 1: la regresión es la línea base"); ax.grid(axis="y", alpha=0); plt.show()

Además del AUC, miremos **qué miran** los modelos. Si dos modelos distintos miran lo mismo, la opacidad de la red no está comprando información nueva.

In [ ]:
reg_p = armar(LogisticRegression(C=1.0, max_iter=3000), prep1).fit(Xtr1, ytr1)
arb_p = armar(modelo_arboles(0), prep1).fit(Xtr1, ytr1)
nombres = [n.split("__")[-1] for n in reg_p.named_steps["prep"].get_feature_names_out()]
top_reg = sorted(zip(nombres, np.abs(reg_p.named_steps["modelo"].coef_[0])), key=lambda t: -t[1])[:3]
print("Las 3 variables que más pesan en la REGRESIÓN :", ", ".join(n for n, _ in top_reg))
if hasattr(arb_p.named_steps["modelo"], "feature_importances_"):
    top_arb = sorted(zip(nombres, arb_p.named_steps["modelo"].feature_importances_), key=lambda t: -t[1])[:3]
    print("Las 3 variables que más pesan en los ÁRBOLES  :", ", ".join(n for n, _ in top_arb))

**Qué mirar:** gana la regresión (0,837) y la red tarda más y varía con la semilla. Los tiempos dependen de la computadora y de lo que esté corriendo en segundo plano: importa la proporción, no los segundos. Las tres variables que más pesan son las mismas en la regresión y en los árboles.

## 5 · Tres variables nuevas

Armamos tres variables derivadas de las que ya existen, con sentido de negocio y **sin usar nada que se sepa después del abandono**:

| Variable | Cómo se calcula | La idea |
| --- | --- | --- |
| `gasto_anual` | ticket promedio × compras de 12 meses | El valor del cliente |
| `contactos_por_compra` | contactos de soporte ÷ (compras + 1) | La fricción por compra |
| `ciclos_sin_comprar` | días desde la última compra × compras ÷ 365 | Qué tan atrasado está respecto de su propio ritmo |

> **Predicción:** ¿cuál de las tres separa mejor a quienes abandonan? ¿Van a mejorar a los modelos?

In [ ]:
def auc_solo(v):
    s = df1[v].fillna(df1[v].median()); a = roc_auc_score(df1.abandono_90d, s); return max(a, 1 - a)
aisladas = pd.DataFrame({"AUC de la variable sola": {v: auc_solo(v) for v in ["dias_desde_ultima_compra", "ticket_promedio", "compras_12m", "contactos_soporte_90d"] + NUEVAS}})
aisladas["tipo"] = ["original" if v not in NUEVAS else "NUEVA" for v in aisladas.index]
display(aisladas.sort_values("AUC de la variable sola", ascending=False))

con = comparar(1, con_nuevas=True)
print()
display(pd.DataFrame({"sin las nuevas": r1["AUC (media)"], "con las tres": con["AUC (media)"], "diferencia": con["AUC (media)"] - r1["AUC (media)"]}))

**Qué mirar:** `gasto_anual` separa tan bien como las originales, pero las tres se parecen a una variable que ya existía, y ningún modelo mejora. La red incluso empeora y se vuelve más inestable. **Una variable puede ser buena sola y no aportar.**

## 6 · La comparación central · Caso 2: NovaRetail 2

Los **mismos 8.000 clientes** y las tres variables nuevas, pero el abandono sale de **otra regla**: tiene un umbral, una interacción y efectos que no son una línea. Esta es la regla, para que se vea que no hay magia:

> **Predicción:** ¿en cuál de los dos casos va a ganar la red?

In [ ]:
# La regla del caso 2: cada línea es un efecto que una regresión no ve
df2 = preparar(2)
gasto = df2.gasto_anual.fillna(df2.gasto_anual.median())
z = (-2.8
     + 0.55 * (df2.segmento == "Esencial") - 0.35 * (df2.segmento == "Premium")     # efecto lineal, como en el caso 1
     + 0.35 * (df2.dias_desde_ultima_compra - 60) / 40                              # efecto lineal, como en el caso 1
     + 2.6 * (df2.ciclos_sin_comprar > 2.0)                                         # UMBRAL: más de 2 ciclos sin comprar
     + 2.4 * ((df2.usa_app == 0) & (df2.contactos_soporte_90d >= 3))                # INTERACCIÓN: sin app y con mucho soporte
     + 1.6 * ((df2.edad < 25) | (df2.edad > 65))                                    # NO ES UNA LÍNEA: jóvenes y mayores
     - 1.2 * (gasto > 600))                                                         # UMBRAL sobre una variable nueva
techo = roc_auc_score(df2.abandono_90d, 1 / (1 + np.exp(-z)))
print(f"Tasa de abandono del caso 2: {c3(100 * df2.abandono_90d.mean(), 1)} %")
print(f"Techo de AUC (con la probabilidad verdadera): {c3(techo)}   ← lo máximo que cualquier modelo podría lograr")

In [ ]:
r2 = comparar(2)
display(r2)

fig, ax = plt.subplots(figsize=(8, 3.4))
orden = list(r2.sort_values("AUC (media)", ascending=False).index)
for i, m in enumerate(orden):
    a, b = r1.loc[m, "AUC (media)"], r2.loc[m, "AUC (media)"]
    ax.plot([a, b], [i, i], color=GRIS, lw=2); ax.scatter([a], [i], s=110, color=GRIS, zorder=3); ax.scatter([b], [i], s=110, color=VIOLETA, zorder=3)
    ax.text(min(a, b) - 0.004, i, c3(min(a, b)), ha="right", va="center"); ax.text(max(a, b) + 0.004, i, c3(max(a, b)), ha="left", va="center")
ax.axvline(techo, color=GRIS, ls="--", lw=1); ax.text(techo, -0.55, "techo del caso 2", ha="center", fontsize=9)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden); ax.invert_yaxis(); ax.set_xlim(0.72, 0.87)
ax.set_xlabel("AUC en la prueba · gris: caso 1 · violeta: caso 2"); ax.grid(axis="y", alpha=0); ax.set_title("La regresión gana en el caso 1 y XGBoost gana en el caso 2"); plt.show()

**Una pregunta más: ¿y si le damos a la regresión «la variable correcta»?** Agregamos dos indicadores armados a mano, uno por cada efecto que la regresión no veía.

> **Predicción:** ¿llega a XGBoost? ¿Supera a la red?

In [ ]:
df2["atrasado"] = (df2.ciclos_sin_comprar > 2.0).astype(int)
df2["sin_app_con_soporte"] = ((df2.usa_app == 0) & (df2.contactos_soporte_90d >= 3)).astype(int)
num_ind = NUM + NUEVAS + ["atrasado", "sin_app_con_soporte"]
Xi, yi = df2[num_ind + CAT], df2["abandono_90d"]
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.25, stratify=yi, random_state=42)
prep_i = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num_ind), ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
auc_ind = roc_auc_score(yi_te, armar(LogisticRegression(max_iter=3000), prep_i).fit(Xi_tr, yi_tr).predict_proba(Xi_te)[:, 1])
print(f"Regresión + 2 indicadores a mano: AUC {c3(auc_ind)}")
print(f"Regresión sola: {c3(r2.iloc[0]['AUC (media)'])} · Red: {c3(r2.iloc[1]['AUC (media)'])} · XGBoost: {c3(r2.iloc[2]['AUC (media)'])}")

**Qué mirar:** en el caso 2 gana XGBoost (0,823, cerca del techo de 0,841), no la red. Una regresión con las dos variables correctas llega a 0,805: supera a las redes y se acerca a XGBoost. **A veces una variable vale más que un modelo más grande.** Y las variables nuevas, que no aportaban en el caso 1, acá sí.

**La conclusión depende de la estructura de los datos:** con relaciones casi lineales alcanza la regresión; con umbrales e interacciones gana un modelo de árboles.

## 7 · Sobreajuste y detención temprana

Una red puede memorizar los datos de entrenamiento. Para verlo, entrenamos una red de 64 y 32 neuronas **época por época** y miramos la pérdida en entrenamiento y en validación.

> **Predicción:** ¿en qué época va a empezar a empeorar la validación?

In [ ]:
Xa, Xb, ya, yb = train_test_split(Xt1, yt1, test_size=0.2, stratify=yt1, random_state=0)
red = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.0, learning_rate_init=0.001, random_state=0)
perdida_tr, perdida_va = [], []
for _ in range(100):
    red.partial_fit(Xa, ya, classes=[0, 1])
    perdida_tr.append(log_loss(ya, red.predict_proba(Xa)[:, 1])); perdida_va.append(log_loss(yb, red.predict_proba(Xb)[:, 1]))
mejor = int(np.argmin(perdida_va))
fig, ax = plt.subplots()
ax.plot(range(1, 101), perdida_tr, color=VERDE, lw=2.4, label="entrenamiento"); ax.plot(range(1, 101), perdida_va, color=ROJO, lw=2.4, label="validación")
ax.axvline(mejor + 1, color="black", ls="--", lw=1.2); ax.text(mejor + 3, max(perdida_va) * 0.97, f"parar acá (época {mejor + 1})")
ax.set_xlabel("Épocas"); ax.set_ylabel("Pérdida"); ax.legend(); plt.show()
print(f"La validación toca fondo en la época {mejor + 1} (pérdida {c3(perdida_va[mejor])}) y en la época 100 está en {c3(perdida_va[-1])}, mientras el entrenamiento sigue bajando ({c3(perdida_tr[-1])}).")

Ahora una **red grande** (256 y 256 neuronas) y **XGBoost muy profundo**, sin freno, y la misma red con **penalización L2** (`alpha`). Los entrenamientos pesados se lanzaron al principio: si todavía no terminaron, la celda espera.

> **Predicción:** ¿qué AUC va a tener la red grande en entrenamiento y cuál en prueba? ¿Y con `alpha = 3`?

In [ ]:
resultados = {a: f.result() for a, f in GRANDES.items()}
filas = [{"modelo": f"Red (256, 256) · alpha {a:g}".replace(".", ","), "AUC entrenamiento": tr, "AUC prueba": te} for a, (tr, te) in resultados.items()]
prof = []
try:
    from xgboost import XGBClassifier
    for s in range(5):
        m = armar(XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=s, n_jobs=1, eval_metric="logloss"), prep1).fit(Xtr1, ytr1)
        prof.append((roc_auc_score(ytr1, m.predict_proba(Xtr1)[:, 1]), roc_auc_score(yte1, m.predict_proba(Xte1)[:, 1])))
    filas.append({"modelo": "XGBoost profundo (300 árboles, prof. 6)", "AUC entrenamiento": np.mean([p[0] for p in prof]), "AUC prueba": np.mean([p[1] for p in prof])})
except ImportError:
    print("XGBoost no está instalado: se omite el árbol profundo.")
filas.append({"modelo": "Regresión logística (referencia)", "AUC entrenamiento": roc_auc_score(yt1, reg.predict_proba(Xt1)[:, 1]), "AUC prueba": roc_auc_score(ye1, reg.predict_proba(Xe1)[:, 1])})
tabla = pd.DataFrame(filas).set_index("modelo"); tabla["brecha"] = tabla["AUC entrenamiento"] - tabla["AUC prueba"]
display(tabla)

fig, ax = plt.subplots(figsize=(7, 3.4)); alphas = list(resultados)
ax.plot(range(len(alphas)), [resultados[a][0] for a in alphas], color=VERDE, lw=2.4, marker="o", label="entrenamiento")
ax.plot(range(len(alphas)), [resultados[a][1] for a in alphas], color=ROJO, lw=2.4, marker="o", label="prueba")
ax.axhline(tabla.loc["Regresión logística (referencia)", "AUC prueba"], color=VIOLETA, ls="--", lw=1.4); ax.text(0.02, 0.842, "regresión logística", color=VIOLETA)
ax.set_xticks(range(len(alphas))); ax.set_xticklabels([f"{a:g}".replace(".", ",") for a in alphas]); ax.set_xlabel("Penalización L2 (alpha)"); ax.set_ylabel("AUC"); ax.legend(); plt.show()

**Qué mirar:** la red grande sin freno llega a 0,992 en entrenamiento y 0,751 en prueba, y XGBoost profundo a 1,000 y 0,792. Con la penalización la brecha se cierra, pero ni así supera a la regresión. La regresión casi no tiene brecha.

## 8 · Arquitecturas · Convolución a mano

Una red convolucional busca formas con **filtros chicos que recorren toda la imagen**. Armamos una imagen de 8×8 con un borde vertical y un filtro de 3×3 que detecta bordes verticales.

> **Predicción:** ¿en qué columnas del mapa de rasgos va a responder el filtro?

In [ ]:
imagen = np.zeros((8, 8)); imagen[:, 4:] = 1.0
filtro = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float)
mapa = np.array([[np.sum(imagen[i:i + 3, j:j + 3] * filtro) for j in range(6)] for i in range(6)])

fig, ejes = plt.subplots(1, 3, figsize=(10, 3))
ejes[0].imshow(imagen, cmap="gray"); ejes[0].set_title("Imagen 8×8"); ejes[1].imshow(filtro, cmap="coolwarm", vmin=-1.5, vmax=1.5); ejes[1].set_title("Filtro 3×3"); ejes[2].imshow(mapa, cmap="viridis"); ejes[2].set_title("Mapa de rasgos")
for ax in ejes: ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()
print("Columnas con respuesta del filtro:", [int(j) + 1 for j in np.where(mapa.max(axis=0) > 0)[0]], "(contando desde 1)")
print(f"Parámetros: capa densa de 784 a 128 = {miles(784 * 128 + 128)} · convolución de 16 filtros de 3×3 = {16 * 9 + 16}")

**Qué mirar:** el filtro se enciende solo donde está el borde, y **el mismo filtro recorre toda la imagen**: por eso 16 filtros tienen 160 parámetros contra 100.480 de una capa densa.

## 9 · Arquitecturas · Secuencias y la fuga del orden

En una serie de tiempo, **el orden importa**. Con una serie semanal sintética (tendencia y estacionalidad) entrenamos el mismo modelo de dos formas: dividiendo **al azar** y dividiendo **por tiempo**.

> **Predicción:** ¿qué división va a dar un error más bajo? ¿Cuál es la honesta?

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
azar_s = np.random.default_rng(7); semanas = np.arange(260)
serie = 200 + 0.6 * semanas + 25 * np.sin(2 * np.pi * semanas / 52) + azar_s.normal(0, 8, 260)
tabla_s = pd.DataFrame({"y": serie})
for k in (1, 2, 3, 52): tabla_s[f"rezago_{k}"] = tabla_s.y.shift(k)
tabla_s = tabla_s.dropna().reset_index(drop=True)
Xs, ys = tabla_s.drop(columns="y").to_numpy(), tabla_s.y.to_numpy(); corte = int(len(tabla_s) * 0.75)
mae_t = np.mean(np.abs(GradientBoostingRegressor(random_state=0).fit(Xs[:corte], ys[:corte]).predict(Xs[corte:]) - ys[corte:]))
orden = np.random.default_rng(0).permutation(len(tabla_s)); tr, te = orden[:corte], orden[corte:]
mae_a = np.mean(np.abs(GradientBoostingRegressor(random_state=0).fit(Xs[tr], ys[tr]).predict(Xs[te]) - ys[te]))
fig, ax = plt.subplots(figsize=(6, 3.4))
barras = ax.bar(["Al azar\n(con fuga)", "Temporal\n(honesto)"], [mae_a, mae_t], color=[ROJO, VERDE], width=0.5)
for b_, v in zip(barras, [mae_a, mae_t]): ax.text(b_.get_x() + b_.get_width() / 2, v + 0.3, c3(v, 1), ha="center", fontweight="bold")
ax.set_ylabel("Error absoluto medio (menos es mejor)"); ax.grid(axis="x", alpha=0); plt.show()
print(f"Dividir al azar da un error {c3(100 * (1 - mae_a / mae_t), 0)} % más bajo, y es una fuga: el modelo ya vio el futuro.")

**Qué mirar:** el error es 8,7 al azar y 15,8 por tiempo. **Un resultado que mejora al mezclar el tiempo es una fuga, no una mejora.**

## 10 · Arquitecturas · Atención a mano

En un transformer, cada palabra **mira a las demás** para entender su contexto. Los pesos de atención salen de comparar las palabras de a pares y normalizar con una **softmax**. Acá usamos vectores de ejemplo armados a mano: **no es un modelo entrenado**, sirve para ver el cálculo.

> **Predicción:** ¿a qué palabra va a mirar más «comprar»?

In [ ]:
palabras = ["el", "cliente", "dejó", "de", "comprar"]
E = np.array([[0.1, 0.0, 0.2], [1.0, 0.1, 0.0], [0.0, 1.0, 0.3], [0.1, 0.1, 0.1], [0.2, 0.9, 1.0]])
Wk = np.array([[1, 0, 0.2], [0, 1, 0.8], [0.2, 0.3, 1]])
puntajes = (E @ np.eye(3)) @ (E @ Wk).T / np.sqrt(3)
atencion = np.exp(puntajes - puntajes.max(axis=1, keepdims=True)); atencion = atencion / atencion.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.2, 4.2)); im = ax.imshow(atencion, cmap="Purples", vmin=0, vmax=0.5)
ax.set_xticks(range(5)); ax.set_xticklabels(palabras); ax.set_yticks(range(5)); ax.set_yticklabels(palabras); ax.set_xlabel("palabra a la que se mira"); ax.set_ylabel("palabra que mira"); ax.grid(False)
for i in range(5):
    for j in range(5): ax.text(j, i, c3(atencion[i, j], 2), ha="center", va="center", color="white" if atencion[i, j] > 0.3 else "black")
plt.show()
print("«comprar» mira a:", {p: c3(v, 2) for p, v in zip(palabras, atencion[4])}, "· cada fila suma", c3(atencion[4].sum(), 1))

**Qué mirar:** «comprar» se mira más a sí misma (0,43) y a «dejó» (0,27). Cada fila suma 1 porque pasa por una softmax.

## 11 · Arquitecturas · Modelos generativos: plausible no es lo mismo que correcto

Un modelo generativo produce texto que **suena bien**. Armamos uno de juguete, que aprende qué palabra suele seguir a cuál a partir de seis reclamos, y lo comparamos con lo que dice «el sistema». Los modelos reales son mucho más sofisticados, pero el problema de fondo es el mismo: **nada garantiza que lo generado sea verdad**.

> **Predicción:** de 200 frases generadas, ¿qué porcentaje va a ser correcto, es decir, coincidir con el sistema?

In [ ]:
import random, re
SISTEMA = {4821: "zapatillas", 4822: "remera", 4823: "mochila", 4824: "campera", 4825: "zapatillas", 4826: "mochila"}
PROBLEMA = {"zapatillas": "llegaron rotas", "remera": "llegó con otro talle", "mochila": "se rompió el cierre", "campera": "llegó tarde"}
corpus = [f"mi pedido {n} de {p} {PROBLEMA[p]}" for n, p in SISTEMA.items()]

siguientes = {}
for frase in corpus:
    palabras_f = ["<inicio>"] + frase.split() + ["<fin>"]
    for a, b in zip(palabras_f, palabras_f[1:]): siguientes.setdefault(a, []).append(b)

def generar(rng):
    actual, salida = "<inicio>", []
    while True:
        actual = rng.choice(siguientes[actual])
        if actual == "<fin>": return " ".join(salida)
        salida.append(actual)

def es_correcta(frase):
    m = re.match(r"mi pedido (\d+) de (\w+) (.*)", frase)
    return bool(m) and SISTEMA.get(int(m.group(1))) == m.group(2) and PROBLEMA.get(m.group(2)) == m.group(3)

rng = random.Random(0); frases = [generar(rng) for _ in range(200)]
correctas = sum(es_correcta(f) for f in frases)
print("Cinco frases generadas:")
for f in frases[:5]: print(("  ✔ " if es_correcta(f) else "  ✘ ") + f)
print(f"\nCorrectas: {correctas} de 200 ({c3(100 * correctas / 200, 0)} %). Todas suenan bien; el resto contradice al sistema.")

**Qué mirar:** todas las frases son gramaticales y suenan seguras, pero la mayoría **inventa** una combinación que el sistema no tiene. Por eso un modelo generativo necesita una evaluación propia y, cuando hay una respuesta oficial, conviene **buscarla y devolverla** en lugar de generarla.

## 12 · Cómo elegir una arquitectura

Todo lo anterior se resume en una función con las preguntas de la clase: ¿qué tipo de dato es?, ¿hay etiquetas?, ¿importa el orden?, ¿hay que generar algo? **La función no decide por vos: te dice por dónde empezar.**

> **Predicción:** ¿qué va a recomendar para un texto con pocos reclamos etiquetados?

In [ ]:
def recomendar(dato, etiquetas=True, pocos_datos=False, orden=False, generar=False):
    if generar:
        return "Modelo generativo (preentrenado)", "Medí respuestas inventadas, datos sensibles y costo por consulta; compará con un buscador de respuestas oficiales."
    if not etiquetas:
        return "Agrupamiento o similitud (sin etiquetas)", "¿Cómo vas a validar sin respuestas marcadas? Pensá en revisión humana de una muestra."
    if dato == "imagen":
        return "Red convolucional" + (" preentrenada y ajustada" if pocos_datos else ""), "Una red desde cero necesita muchas imágenes; con pocas, reutilizá un modelo preentrenado."
    if dato == "texto":
        return "Transformer" + (" preentrenado y ajustado" if pocos_datos else ""), "Compará con regresión sobre el conteo de palabras: es tu línea base."
    if orden:
        return "Árboles con variables de rezago (red recurrente como candidata)", "Dividí por tiempo, no al azar, y compará con repetir el período anterior."
    return "Regresión regularizada y después árboles (la red solo si compara bien)", "En tablas, la regresión y los árboles suelen alcanzar; compará con varias semillas."

casos = [("Fotos de productos dañados", dict(dato="imagen", pocos_datos=True)), ("Reclamos escritos, pocos etiquetados", dict(dato="texto", pocos_datos=True)), ("Ventas semanales por producto", dict(dato="tabular", orden=True)),
         ("Abandono de clientes", dict(dato="tabular")), ("Resumir hilos de atención", dict(dato="texto", generar=True)), ("Segmentar clientes sin grupos definidos", dict(dato="tabular", etiquetas=False))]
for nombre, args in casos:
    familia, aviso = recomendar(**args)
    print(f"{nombre}\n   Empezá por: {familia}\n   Ojo: {aviso}\n")
print("En todos los casos: una línea base primero y varias semillas.")

## 13 · Laboratorio de errores

Cinco errores típicos, cada uno con **el número que lo esconde** y **el número honesto**. Los calculamos con lo que ya corrimos.

> **Predicción:** ¿cuál de los cinco errores cambia más la conclusión?

In [ ]:
red_pequena = [roc_auc_score(yte1, armar(MLPClassifier(hidden_layer_sizes=(8,), early_stopping=True, validation_fraction=0.15, n_iter_no_change=10, max_iter=500, random_state=s), prep1).fit(Xtr1, ytr1).predict_proba(Xte1)[:, 1]) for s in range(5)]
red_media, base_auc = r1.iloc[1]["AUC (media)"], r1.iloc[0]["AUC (media)"]
errores = pd.DataFrame([
    ("1 · Sin línea base", f"La red da {c3(red_media)}: parece un buen resultado", f"La regresión da {c3(base_auc)}: la red no la supera"),
    ("2 · Mezclar el tiempo", f"Error {c3(mae_a, 1)} al dividir al azar", f"Error {c3(mae_t, 1)} con la división temporal"),
    ("3 · Mirar solo el entrenamiento", f"AUC {c3(resultados[0.0][0])} en entrenamiento", f"AUC {c3(resultados[0.0][1])} en prueba"),
    ("4 · Una sola semilla", f"La mejor semilla de una red chica da {c3(max(red_pequena))}", f"Entre semillas va de {c3(min(red_pequena))} a {c3(max(red_pequena))} (media {c3(np.mean(red_pequena))})"),
    ("5 · Omitir el costo", f"La red logra {c3(red_media)}", f"Entrena unas {c3(r1.iloc[1]['entrena en (s)'] / r1.iloc[0]['entrena en (s)'], 0)} veces más lento, con {miles(parametros(21, (64, 32)))} parámetros contra 22, y no supera a la regresión"),
], columns=["error", "el número que lo esconde", "el número honesto"]).set_index("error")
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)
display(errores)

**Qué mirar:** en los cinco, el número engañoso parece bueno hasta que se lo compara con la línea base, con el tiempo o con las semillas. La defensa es siempre la misma: **una línea base, la misma partición y varias semillas**.

## 14 · Cierre: la respuesta a la pregunta

**¿Cuándo una red neuronal mejora una solución y cuándo solo agrega costo y opacidad?**

In [ ]:
cierre = pd.DataFrame({"Caso 1 (casi lineal)": r1["AUC (media)"], "Caso 2 (umbrales e interacciones)": r2["AUC (media)"]})
cierre.loc["Regresión + 2 indicadores a mano"] = [np.nan, auc_ind]
display(cierre)
print("Gana en el caso 1 :", cierre["Caso 1 (casi lineal)"].idxmax())
print("Gana en el caso 2 :", cierre["Caso 2 (umbrales e interacciones)"].iloc[:3].idxmax())

**Cuatro ideas para llevarse**

1. **Primero la línea base.** Una red se mide contra algo más simple, no contra nada.
2. **Depende de la estructura de los datos.** Con relaciones casi lineales alcanza la regresión; con umbrales e interacciones gana un modelo de árboles.
3. **En tablas, probá árboles antes que una red.** Las redes se justifican sobre todo con imagen, secuencia y texto.
4. **Costo, explicabilidad y riesgo también cuentan.**

**Ticket de salida** (una frase por cada uno): qué voy a repetir · qué voy a verificar · qué voy a dejar de hacer.

Seguimos con la **práctica guiada**: `clase_3_practica_guiada.ipynb`.